# Random Forest and XGBoost on Descriptors & Fragment Features (Representation A)

This notebook uses features set A (refer to notebook 1a) and selects hyperparameters and models using k-fold cross-validation

**Goal:** Select the best performing model/hyperparameters through k fold cross validations using the chemical features extracted in the previous notebook.


### Import Required Libraries

- **pandas/numpy**: Data manipulation
- **seaborn/matplotlib**: Data visualization
- **sklearn**: Machine learning tools (models, cross-validation, metrics)
- **ydata_profiling**: Data profiling and analysis


In [16]:
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

#ignore warnings
import warnings
warnings.filterwarnings('ignore')

## Step 1: Load Feature Data

Load the training features extracted in the previous notebook (`training_features.csv`).

This dataset contains ~163 feature columns plus the target variable (ACTIVE).


In [9]:
features_df = pd.read_csv("training_features.csv")
features_df.head()


,Unnamed: 0,INDEX,SMILES,ACTIVE,MolFromSmiles,NumAtoms,NumHeavyAtoms,NumBonds,fr_Al_COO,fr_Al_OH,...,CalcNumRings,CalcNumRotatableBonds,CalcNumSaturatedCarbocycles,CalcNumSaturatedHeterocycles,CalcNumSaturatedRings,CalcNumSpiroAtoms,CalcNumUnspecifiedAtomStereoCenters,CalcPhi,CalcTPSA,_CalcMolWt
0,0,1,O=C(Nc1ccc2c(c1)OCCO2)C1CCN(c2ncccn2)CC1,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b220>,25,25,28,0,0,...,4,3,0,1,1,0,0,4.368063,76.58,340.383
1,1,2,COCCCN1C(=O)C2C(C(=O)Nc3cccc(Cl)c3)C3C=CC2(O3)...,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc38b30>,35,35,39,0,0,...,5,8,1,2,3,1,5,6.877647,96.97,502.011
2,2,3,CCSc1ncc(Cl)c(C(=O)Nc2ccccc2C)n1,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b0d0>,20,20,21,0,0,...,2,4,0,0,0,0,0,4.977937,54.88,307.806
3,3,4,COc1ccc2cc(/C=N/NC(=O)CN(c3ccccc3C)S(=O)(=O)c3...,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b140>,36,36,39,0,0,...,4,8,0,0,0,0,0,7.516779,100.96,523.014
4,4,5,CCCC(=O)Nc1nc2ccc(NC(=O)c3c(F)c(F)c(OC)c(F)c3F...,0.0,<rdkit.Chem.rdchem.Mol object at 0x12dc3b1b0>,30,30,32,0,0,...,3,6,0,0,0,0,0,6.202841,80.32,441.406


## Step 2 - Split into X (features) and y (targets)

**Separate features (X) from target (y)**: 
   - X: All feature columns (excluding ACTIVE, SMILES, INDEX, MolFromSmiles)
   - y: ACTIVE column (0 = inactive, 1 = active)

In [10]:
features_df.columns

Index(['Unnamed: 0', 'INDEX', 'SMILES', 'ACTIVE', 'MolFromSmiles', 'NumAtoms',
       'NumHeavyAtoms', 'NumBonds', 'fr_Al_COO', 'fr_Al_OH',
       ...
       'CalcNumRings', 'CalcNumRotatableBonds', 'CalcNumSaturatedCarbocycles',
       'CalcNumSaturatedHeterocycles', 'CalcNumSaturatedRings',
       'CalcNumSpiroAtoms', 'CalcNumUnspecifiedAtomStereoCenters', 'CalcPhi',
       'CalcTPSA', '_CalcMolWt'],
      dtype='object', length=164)

## Step 3: Train Machine Learning Model

Train a **Random Forest Classifier** to predict molecular activity.

**Process:**

1. **10-Fold Cross-Validation**: 
   - Split data into 10 folds - use StratifiedKFold to distribute classes better
   - Train on 9 folds, test on 1 fold
   - Repeat 10 times with different splits
   - This gives a robust estimate of model performance

2. **XGBoost Classifier**
   - Select hyperparameters to experiment on
   - Apply to k-folds
   - Good for tabular data

3. **Random Forest Classifier**:
   - Ensemble method that combines multiple decision trees
   - Good baseline model for structured data
   - Handles many features well

**Output:** Mean accuracy across all 10 folds (~95.3% in this case)


In [11]:
TARGET_COL = "ACTIVE"  
y = features_df[TARGET_COL]

In [12]:
non_feature_cols = [TARGET_COL, "SMILES", "MolFromSmiles", "Unnamed: 0", "INDEX"]
non_feature_cols = [c for c in non_feature_cols if c in features_df.columns]


y = features_df[TARGET_COL]

X = features_df.drop(columns=non_feature_cols)
X = X.select_dtypes(include=[np.number])

X.shape, y.shape

((202895, 159), (202895,))

Set random seed for reproducibility. This ensures that results are consistent across runs.


In [13]:
seed = 20231124
np.random.seed(seed)

A comparison between XGBoost and Random Forest
- we create a helper function to get a confidence interval in the resulting models

In [17]:
def calculate_ci(scores, confidence=0.95):
    n = len(scores)
    mean = np.mean(scores)
    std_err = stats.sem(scores)  # Standard error of the mean
    ci = stats.t.interval(confidence, n-1, loc=mean, scale=std_err) # uses t test
    return mean, ci

## 3a - XGBoost hyperparam exploration on Kfolds

In [ ]:
## XGB things
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)
param_dist_xgb = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_base = XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1   
)

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_xgb,
    n_iter=20,
    scoring="roc_auc",
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)


xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1  
)


xgb_search.fit(X, y)

auc_scores = cross_val_score(
    xgb_clf,
    X,
    y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=1  
)

print("AUC per fold:", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Std AUC:", auc_scores.std())


print("Best XGBoost AUC:", xgb_search.best_score_)
print("Best XGBoost params:", xgb_search.best_params_)


Fitting 10 folds for each of 20 candidates, totalling 200 fits
AUC per fold: [0.8988179  0.88927442 0.90297625 0.88985802 0.90256887 0.89608164
 0.88475097 0.88595345 0.88826548 0.88673744]
Mean AUC: 0.8925284438374638
Std AUC: 0.0065977073606515255
Best XGBoost AUC: 0.9022729279222063
Best XGBoost params: {'subsample': 1.0, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}


# 3b - Random Forest Hyperparams exploration on kfolds 

In [ ]:
rf_base = RandomForestClassifier(
    n_jobs=-1,
    random_state=42
)

param_dist_rf = {
    "n_estimators": [100, 200, 300, 400],
    "max_depth": [10, 30, 50],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    # "bootstrap": [True]
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist_rf,
    n_iter=10,
    scoring="roc_auc",
    cv=cv, # same cross-fold
    n_jobs=-1
)


rf_search.fit(X, y)

auc_scores = cross_val_score(
    rf_base,
    X,
    y,
    cv=cv,
    scoring="roc_auc"
)
print("AUC per fold:", auc_scores)
print("Mean AUC:", auc_scores.mean())
print("Std AUC:", auc_scores.std())


print("Best RF AUC:", rf_search.best_score_)
print("Best RF params:", rf_search.best_params_)


Fitting 10 folds for each of 10 candidates, totalling 100 fits
AUC per fold: [0.89433618 0.88619681 0.89602748 0.88755804 0.90187275 0.89561073
 0.88489675 0.88873708 0.88273982 0.88765714]
Mean AUC: 0.8905632779882031
Std AUC: 0.005750368708265602
Best RF AUC: 0.8999472791583448
Best RF params: {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 30, 'bootstrap': True}


# 3c - Based on chosen hyperparams, which is the best (using kfolds again)

**XGBoost** 
- Best XGBoost AUC: 0.9022729279222063
- Best XGBoost params: {'subsample': 1.0, 'n_estimators': 400, 'min_child_weight': 5, 'max_depth': 8, 'learning_rate': 0.1, 'colsample_bytree': 0.8}

**Random Forest**
- Best RF AUC: 0.8999472791583448
- Best RF params: {'n_estimators': 400, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 30, 'bootstrap': True}

Note: will be removing max depth since this was only done to reduce runtime 😅

In [28]:

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_clf = XGBClassifier(
    n_estimators=400,
    subsample=1.0,
    colsample_bytree=0.8,
    min_child_weight= 5,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    n_jobs=-1
)

rf_clf = RandomForestClassifier(
    n_estimators=300,
    n_jobs=-1,
    random_state=42,
    max_features="sqrt",
    min_samples_leaf=2,
    # bootstrap=True
)

xgb_auc = cross_val_score(
    xgb_clf, X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

rf_auc = cross_val_score(
    rf_clf, X, y,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

# Calculate metrics with confidence intervals
xgb_mean, xgb_ci = calculate_ci(xgb_auc)
rf_mean, rf_ci = calculate_ci(rf_auc)

# display comparison
print("XGBoost results: ")
print("AUC per fold:", np.round(xgb_auc, 4))
print(f"Mean AUC: {xgb_mean:.4f}")
print(f"Std AUC: {xgb_auc.std():.4f}")
print(f"95% CI: [{xgb_ci[0]:.4f}, {xgb_ci[1]:.4f}]")

print("*******")
print("Random Forest results: ")
print("AUC per fold:", np.round(rf_auc, 4))
print(f"Mean AUC: {rf_mean:.4f}")
print(f"Std AUC: {rf_auc.std():.4f}")
print(f"95% CI: [{rf_ci[0]:.4f}, {rf_ci[1]:.4f}]")

# use t-test to compare auc
t_stat, p_value = stats.ttest_rel(xgb_auc, rf_auc)

XGBoost results: 
AUC per fold: [0.8888 0.8948 0.8934 0.8812 0.8869]
Mean AUC: 0.8890
Std AUC: 0.0049
95% CI: [0.8823, 0.8958]
*******
Random Forest results: 
AUC per fold: [0.8959 0.8998 0.9048 0.8892 0.892 ]
Mean AUC: 0.8963
Std AUC: 0.0055
95% CI: [0.8887, 0.9040]


In [29]:
# quick compare stats to find out (based on the hyperparams chosen) which is the better model
t_stat, p_value = stats.ttest_rel(xgb_auc, rf_auc)
print(f"Difference in means: {abs(xgb_mean - rf_mean):.4f}")
print(f"Paired t-test p-value: {p_value:.4f}")
if p_value < 0.05:
    winner = "XGBoost" if xgb_mean > rf_mean else "Random Forest"
    print(f"Winner: {winner} (statistically significant)")
else:
    print("No statistically significant difference between models")


Difference in means: 0.0073
Paired t-test p-value: 0.0034
Winner: Random Forest (statistically significant)
